# Adversarial attacks on face recognition: perceptual, statistical and embedding-space analysis
Runs the complete pipeline: MTCNN → FaceNet (VGGFace2) → 11 attacks + 3 controls → metrics → tables & figures.

The code runs on CPU (that is how the reported results were produced). Expect ~1–1.5 h per dataset on a standard Colab runtime; Square and NES are the slowest steps.
LPIPS is computed automatically here (its weights download from the internet).

In [ ]:
!git clone -q https://github.com/Nainajha09/Adversarial-Attack-on-Face-Recognition-Model repo
%cd repo/advface
!pip install -q --no-deps facenet-pytorch==2.6.0
!pip install -q onnxruntime pytorch-fid lpips scikit-image
!bash download_weights.sh

## LFW
Downloads exactly the 500 photos listed in `data/lfw_selection.txt`.

In [ ]:
!bash get_lfw.sh
!bash run_all.sh lfw lfw_raw

## CelebA (optional)
Upload `img_align_celeba/` and `identity_CelebA.txt` to Google Drive and set the paths below.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CELEBA_IMG = '/content/drive/MyDrive/celeba/img_align_celeba'
CELEBA_ID  = '/content/drive/MyDrive/celeba/identity_CelebA.txt'
!bash run_all.sh celeba {CELEBA_IMG} {CELEBA_ID}

## Results

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image
for ds in ['lfw', 'celeba']:
    for t in sorted(glob.glob(f'results/{ds}/T*.csv')):
        print(ds, t.split('/')[-1]); display(pd.read_csv(t))
for f in sorted(glob.glob('results/lfw/F*.png')): display(Image(f, width=700))

## Dataset comparison (after both datasets are run)
Mann–Whitney U test per attack and metric (independent samples).

In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu
a, b = pd.read_csv('results/lfw_per_image.csv'), pd.read_csv('results/celeba_per_image.csv')
rows = []
for atk in sorted(set(a.attack) & set(b.attack)):
    for m in ['SSIM', 'PSNR', 'LPIPS', 'NMI', 'self_dist']:
        x, y = a[a.attack == atk][m].dropna(), b[b.attack == atk][m].dropna()
        if len(x) and len(y):
            rows.append(dict(attack=atk, metric=m, LFW=x.mean(), CelebA=y.mean(), p=mannwhitneyu(x, y).pvalue))
display(pd.DataFrame(rows).round(4))